# FitRank — Cross-Encoder Fine-Tuning

Fine-tunes `cross-encoder/ms-marco-MiniLM-L-6-v2` on resume-job relevance pairs.

Requires a GPU runtime: **Runtime → Change runtime type → T4 GPU**.

In [ ]:
!pip install -q -U sentence-transformers datasets

In [ ]:
import torch
print("GPU available:", torch.cuda.is_available())
print("Device:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")

## 1. Load data
Upload `train_pairs.csv` when prompted.

In [ ]:
from google.colab import files
uploaded = files.upload()

In [ ]:
import pandas as pd
from datasets import Dataset

df = pd.read_csv("train_pairs.csv")
print(f"Loaded {len(df)} pairs")
print(df["label"].value_counts())

def truncate_words(text, max_words=250):
    # matches the truncation policy used at inference time -- keeps
    # train/serve behavior consistent
    return " ".join(str(text).split()[:max_words])

df["resume_text_trunc"] = df["resume_text"].apply(truncate_words)

# small slice held out purely to monitor training loss (not the real eval --
# that happens separately, locally, on an independently labeled set)
df = df.sample(frac=1, random_state=42).reset_index(drop=True)
n_val = max(1, int(len(df) * 0.05))
val_df = df.iloc[:n_val]
train_df = df.iloc[n_val:]

train_dataset = Dataset.from_dict({
    "query": train_df["resume_text_trunc"].tolist(),
    "response": train_df["job_text"].tolist(),
    "label": train_df["label"].astype(float).tolist(),
})
val_dataset = Dataset.from_dict({
    "query": val_df["resume_text_trunc"].tolist(),
    "response": val_df["job_text"].tolist(),
    "label": val_df["label"].astype(float).tolist(),
})

print(f"Train: {len(train_dataset)}  |  Val (monitoring only): {len(val_dataset)}")

## 2. Model and training setup

In [ ]:
from sentence_transformers.cross_encoder import (
    CrossEncoder,
    CrossEncoderTrainer,
    CrossEncoderTrainingArguments,
)
from sentence_transformers.cross_encoder.losses import BinaryCrossEntropyLoss
from sentence_transformers.cross_encoder.evaluation import CrossEncoderClassificationEvaluator

MODEL_NAME = "cross-encoder/ms-marco-MiniLM-L-6-v2"
OUTPUT_DIR = "fitrank-cross-encoder"

model = CrossEncoder(MODEL_NAME, num_labels=1, max_length=512)
loss = BinaryCrossEntropyLoss(model)

evaluator = CrossEncoderClassificationEvaluator(
    sentence_pairs=list(zip(val_dataset["query"], val_dataset["response"])),
    labels=val_dataset["label"],
    name="val-monitoring",
)

args = CrossEncoderTrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=3,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    learning_rate=2e-5,
    warmup_ratio=0.1,
    fp16=torch.cuda.is_available(),
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=2,
    logging_steps=25,
    load_best_model_at_end=True,
    metric_for_best_model="eval_val-monitoring_accuracy",
)

trainer = CrossEncoderTrainer(
    model=model,
    args=args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    loss=loss,
    evaluator=evaluator,
)

## 3. Train

In [ ]:
trainer.train()

## 4. Save and download

In [ ]:
model.save_pretrained(OUTPUT_DIR)
print(f"Saved to ./{OUTPUT_DIR}")

In [ ]:
import shutil
shutil.make_archive(OUTPUT_DIR, "zip", OUTPUT_DIR)
files.download(f"{OUTPUT_DIR}.zip")